# 06 — Combined Built-in + MCP Tools

Shows how to combine **built-in tools** (Calculator, Clock) with **MCP tools** (filesystem)  
and run a multi-step agent loop that uses both.

**Prerequisites**: `npx` on PATH and `OPENAI_API_KEY` set.

In [1]:
import asyncio, json
from agent_framework.extensions.mcp import MCPClient, MCPTool
from agent_framework.core.tools.builtin_tools import CalculatorTool, GetCurrentTimeTool
from agent_framework.providers.llm.openai.openai_client import OpenAIClient
from agent_framework.core.memory.unbounded_memory import UnboundedMemory
from agent_framework.core.messages.client_messages import UserMessage, SystemMessage, ToolExecutionResultMessage

In [3]:
SSE_URL = 'http://localhost:9000/sse'  # MCP server: docker compose --profile mcp up -d mcp-server

async def demo():
    builtin_tools = [CalculatorTool(), GetCurrentTimeTool()]
    print(f'Built-in tools: {[t.name for t in builtin_tools]}')

    mcp_client = MCPClient()
    try:
        await mcp_client.connect_sse(SSE_URL)
        mcp_tools = await MCPTool.from_mcp_client(mcp_client)
        print(f'MCP tools: {[t.name for t in mcp_tools]}')

        all_tools = builtin_tools + mcp_tools
        print(f'Total: {len(all_tools)} tools')

        client = OpenAIClient(model='gpt-4o')
        memory = UnboundedMemory()
        memory.add_message(SystemMessage(content='You have calculator, clock, and filesystem tools. Use them.'))
        memory.add_message(UserMessage(content=[{'type': 'text', 'text': 'Calculate 123 * 456 and save the result to /tmp/calculation.txt'}]))

        for i in range(5):
            print(f'\n--- Iteration {i+1} ---')
            # get_openai_schema() for OpenAI client; get_schema() for ReActAgent
            response = await client.generate(
                messages=memory.get_messages(),
                tools=[t.get_openai_schema() for t in all_tools],
            )
            if not response.tool_calls:
                print(f'Final answer: {response.content}')
                break
            memory.add_message(response)
            for tc in response.tool_calls:
                # ToolCallMessage: tc.name, tc.arguments (already dict) — not tc.function[...]
                name = tc.name
                args = tc.arguments if isinstance(tc.arguments, dict) else json.loads(tc.arguments)
                print(f'Calling: {name}({args})')
                tool = next((t for t in all_tools if t.name == name), None)
                if tool:
                    res = await tool.execute(**args)
                    result_text = str(res.content)[:100]
                    print(f'Result: {result_text}...')
                    memory.add_message(ToolExecutionResultMessage(
                        content=result_text, tool_call_id=tc.id, name=name
                    ))

    except Exception as e:
        print(f'⚠ Error: {e}')
        print('  Requires: Node.js + npx  (npm install -g npx)')
    finally:
        if mcp_client.is_connected:
            await mcp_client.disconnect()

await demo()


Built-in tools: ['calculator', 'get_current_time']
MCP tools: ['add', 'subtract', 'multiply', 'echo', 'to_uppercase', 'word_count', 'server_info']
Total: 9 tools

--- Iteration 1 ---
Calling: calculator({'expression': '123 * 456'})
Result: [{'type': 'text', 'text': '{"result": 56088, "expression": "123 * 456"}'}]...

--- Iteration 2 ---
Calling: echo({'message': '56088'})
Result: [{'type': 'text', 'text': '56088'}]...
Calling: server_info({})
Result: [{'type': 'text', 'text': '{"name":"agent-framework-demo","transport":"sse","tools":["add","subtract...

--- Iteration 3 ---
Calling: echo({'message': 'The result of 123 * 456 is 56088. Now saving to /tmp/calculation.txt.'})
Result: [{'type': 'text', 'text': 'The result of 123 * 456 is 56088. Now saving to /tmp/calculation.txt.'}]...

--- Iteration 4 ---
Final answer: ['I don\'t have direct access to a filesystem to save files. You can manually save the result "56088" to `/tmp/calculation.txt` on your system.']
